# ESimCSE on STSB

Official paper: https://aclanthology.org/2022.coling-1.342/  
Official code: https://github.com/caskcsg/sentemb/tree/main/ESimCSE  
Checkpoint: `ffgcc/esimcse-bert-base-uncased`  
Dataset source: mteb/stsbenchmark-sts (https://aclanthology.org/S17-2001/)

This notebook **re-evaluates the authors' released pretrained checkpoint** on one public test set. It does not retrain the original method. It uses a common direct-cosine protocol: encode each sentence independently, compute pairwise cosine similarity, then compare with human scores using Spearman and Pearson correlation. It requires an NVIDIA CUDA GPU and keeps all code in this notebook. Its outputs are empty until you run all cells on the RTX 3080.


In [1]:
# Install only missing packages; preserve an existing CUDA-enabled PyTorch build.
import importlib.util
import subprocess
import sys

requirements = {
    "torch": "torch>=2.2,<3",
    "transformers": "transformers==4.46.3",
    "datasets": "datasets==3.2.0",
    "huggingface_hub": "huggingface_hub>=0.26,<1",
    "numpy": "numpy>=1.26,<3",
    "scipy": "scipy>=1.11,<2",
    "pandas": "pandas>=2.1,<3",
}
missing = [spec for module, spec in requirements.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

import importlib.metadata as metadata
for module in requirements:
    package = "huggingface-hub" if module == "huggingface_hub" else module
    print(f"{package}=={metadata.version(package)}")


torch==2.1.2+cu118
transformers==4.46.3
datasets==3.2.0
huggingface-hub==0.36.2
numpy==1.23.5
scipy==1.15.3
pandas==2.3.3


In [2]:
import random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import HfApi
from scipy.stats import pearsonr, spearmanr
from transformers import AutoModel, AutoTokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

if not torch.cuda.is_available():
    raise RuntimeError("This notebook is configured for a CUDA GPU. Check the RTX 3080 driver and CUDA-enabled PyTorch installation.")
DEVICE = torch.device("cuda:0")
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA runtime:", torch.version.cuda)
print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))

# Batch size 32 and dynamic padding fit typical 10/12 GiB RTX 3080 cards.
BATCH_SIZE = 32
MAX_LENGTH = 512


GPU: NVIDIA GeForce RTX 3080 Ti
CUDA runtime: 11.8
GPU memory (GiB): 11.63


In [3]:
METHOD_NAME = 'ESimCSE'
DATASET_NAME = 'STSB'
MODEL_ID = 'ffgcc/esimcse-bert-base-uncased'
POOLING = 'cls_before_pooler'
EXPECTED_TEST_ROWS = 1379
LABEL_RANGE = (0.0, 5.0)


In [4]:
if DATASET_NAME == "STSB":
    raw = load_dataset("mteb/stsbenchmark-sts", split="test")
    frame = pd.DataFrame({
        "sentence1": raw["sentence1"],
        "sentence2": raw["sentence2"],
        "gold_score": raw["score"],
    })
else:
    # JSICK mirrors the original English SICK sentences/scores and preserves the standard test split.
    raw = load_dataset("hpprc/jsick", "base", split="test")
    frame = pd.DataFrame({
        "sentence1": raw["premise_en"],
        "sentence2": raw["hypothesis_en"],
        "gold_score": raw["score_en"],
    })

assert len(frame) == EXPECTED_TEST_ROWS, f"Unexpected test set size: {len(frame)}"
assert frame.notna().all().all(), "Dataset contains missing values"
assert frame["gold_score"].between(*LABEL_RANGE).all(), "Score outside documented range"
assert frame["sentence1"].str.len().gt(0).all()
assert frame["sentence2"].str.len().gt(0).all()
print("Dataset:", DATASET_NAME, "| test pairs:", len(frame))
print("Gold score range:", (frame.gold_score.min(), frame.gold_score.max()))
display(frame.head(5))


Dataset: STSB | test pairs: 1379
Gold score range: (0.0, 5.0)


,sentence1,sentence2,gold_score
0,A girl is styling her hair.,A girl is brushing her hair.,2.5
1,A group of men play soccer on the beach.,A group of boys are playing soccer on the beach.,3.6
2,One woman is measuring another woman's ankle.,A woman measures another woman's ankle.,5.0
3,A man is cutting up a cucumber.,A man is slicing a cucumber.,4.2
4,A man is playing a harp.,A man is playing a keyboard.,1.5


In [5]:
api = HfApi()
model_sha = api.model_info(MODEL_ID).sha
dataset_repo = "mteb/stsbenchmark-sts" if DATASET_NAME == "STSB" else "hpprc/jsick"
dataset_sha = api.dataset_info(dataset_repo).sha
print("Model:", MODEL_ID, "| revision:", model_sha)
print("Dataset:", dataset_repo, "| revision:", dataset_sha)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=model_sha)
model = AutoModel.from_pretrained(MODEL_ID, revision=model_sha).to(DEVICE)
model.eval()
assert model.config.hidden_size == 768, "Expected a BERT-base-sized checkpoint"


Model: ffgcc/esimcse-bert-base-uncased | revision: b46170e5aeab96c02dfeeabeecb82abfc3ead097
Dataset: mteb/stsbenchmark-sts | revision: 96943a16ea6a35129e253c659081cb59daf81b30


pytorch_model.bin:   0%|          | 0.00/878M [00:00<?, ?B/s]

/root/miniconda3/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


In [6]:
def render_text(text):
    if POOLING != "prompt_mask":
        return text
    # Matches the author's unsup-bert evaluation template after placeholder expansion.
    # The original evaluator adds a period when the source has no terminal punctuation.
    if text and text[-1] not in '.?"\'':
        text += "."
    return f'This sentence of "{text}" means{tokenizer.mask_token}.'


@torch.inference_mode()
def encode_sentences(texts):
    vectors = []
    for start in range(0, len(texts), BATCH_SIZE):
        batch_texts = [render_text(text) for text in texts[start:start + BATCH_SIZE]]
        batch = tokenizer(batch_texts, padding=True, truncation=True,
                          max_length=MAX_LENGTH, return_tensors="pt").to(DEVICE)
        outputs = model(**batch, return_dict=True)
        if POOLING == "cls_before_pooler":
            embeddings = outputs.last_hidden_state[:, 0, :]
        elif POOLING == "cls_pooler":
            embeddings = outputs.pooler_output
            if embeddings is None:
                raise RuntimeError("This checkpoint has no CLS pooler output")
        elif POOLING == "prompt_mask":
            mask_positions = batch["input_ids"].eq(tokenizer.mask_token_id)
            if not torch.all(mask_positions.sum(dim=1) == 1):
                raise RuntimeError("Prompt template must contain exactly one untruncated [MASK]")
            embeddings = outputs.last_hidden_state[mask_positions]
        else:
            raise ValueError(POOLING)
        vectors.append(F.normalize(embeddings.float(), p=2, dim=1).cpu())
    return torch.cat(vectors).numpy()


# Encode each distinct sentence once. This does not use any gold labels.
unique_sentences = list(dict.fromkeys(frame.sentence1.tolist() + frame.sentence2.tolist()))
print("Distinct sentences to encode:", len(unique_sentences))
vectors = encode_sentences(unique_sentences)
vector_by_sentence = dict(zip(unique_sentences, vectors))
left = np.stack([vector_by_sentence[text] for text in frame.sentence1])
right = np.stack([vector_by_sentence[text] for text in frame.sentence2])
frame["cosine_similarity"] = np.sum(left * right, axis=1)
assert np.isfinite(frame["cosine_similarity"]).all()


Distinct sentences to encode: 2552


In [7]:
gold = frame["gold_score"].to_numpy(dtype=float)
pred = frame["cosine_similarity"].to_numpy(dtype=float)
spearman = spearmanr(gold, pred).statistic
pearson = pearsonr(gold, pred).statistic
summary = pd.DataFrame([{
    "method": METHOD_NAME,
    "dataset": DATASET_NAME,
    "test_pairs": len(frame),
    "spearman": float(spearman),
    "pearson": float(pearson),
    "model_id": MODEL_ID,
    "model_revision": model_sha,
    "dataset_revision": dataset_sha,
    "pooling": POOLING,
    "max_length": MAX_LENGTH,
    "batch_size": BATCH_SIZE,
    "seed": SEED,
}])
display(summary)
print("Examples with low model similarity:")
display(frame.sort_values("cosine_similarity").head(5))
print("Examples with high model similarity:")
display(frame.sort_values("cosine_similarity", ascending=False).head(5))


,method,dataset,test_pairs,spearman,pearson,model_id,model_revision,dataset_revision,pooling,max_length,batch_size,seed
0,ESimCSE,STSB,1379,0.80148,0.804252,ffgcc/esimcse-bert-base-uncased,b46170e5aeab96c02dfeeabeecb82abfc3ead097,96943a16ea6a35129e253c659081cb59daf81b30,cls_before_pooler,512,32,42


Examples with low model similarity:


,sentence1,sentence2,gold_score,cosine_similarity
237,A man is smoking.,A baby is sucking on a pacifier.,0.0,0.272027
445,A brown and white dog is running across a brow...,A person hanging from a rocky cliff.,0.0,0.326308
442,A dog standing in the water.,A girl playing on a parachute.,0.4,0.347465
115,A boy is playing violin on stage.,A person is mixing a pot.,0.0,0.348539
231,A woman is cutting an onion.,A horse is standing.,0.0,0.355210


Examples with high model similarity:


,sentence1,sentence2,gold_score,cosine_similarity
330,Train in a station.,a train in a station.,5.0,0.993002
664,It's not a good idea.,It is not a good idea.,5.0,0.992445
623,A brown dog is jumping.,A brown dog is jumping,5.0,0.992427
854,Maybe this could work for you.,Maybe this can work for you.,5.0,0.992336
125,A woman and man are dancing in the rain.,A man and woman are dancing in rain.,5.0,0.991908
